# Semantic I/O keystone demo

This deterministic notebook demonstrates the Sprint B keystone cascade: image stimuli, decoded classifier output, `Trace.model_profile`, saved activation subsets, MDS evolution annotations, MDS thumbnail nodes, and raw-input display.

In [ ]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
import sys
import tempfile
import warnings

import numpy as np
from PIL import Image, ImageDraw
import torch
from torch import nn

repo_root = Path.cwd().resolve()
if not (repo_root / "torchlens").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import torchlens as tl  # noqa: E402
from torchlens.data_classes.trace import ResolvedPreprocessing  # noqa: E402
from torchlens.intervention.errors import MultiMatchWarning  # noqa: E402

torch.manual_seed(2024)
warnings.filterwarnings("ignore", category=MultiMatchWarning)

OUT = Path(tempfile.mkdtemp(prefix="torchlens_semantic_io_keystone_"))
print(f"torchlens {tl.__version__}; artifacts={OUT}")

In [ ]:
class TinyImageClassifier(nn.Module):
    """Small deterministic image classifier with label metadata."""

    def __init__(self) -> None:
        """Initialize fixed convolutional blocks and class labels."""

        super().__init__()
        self.config = SimpleNamespace(
            id2label={0: "vertical", 1: "horizontal", 2: "diagonal"},
            num_labels=3,
        )
        self.block1 = nn.Sequential(nn.Conv2d(3, 4, kernel_size=3, padding=1), nn.ReLU())
        self.block2 = nn.Sequential(nn.Conv2d(4, 6, kernel_size=3, padding=1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(6, 3)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        """Populate deterministic feature detectors and classifier weights."""

        with torch.no_grad():
            self.block1[0].weight.zero_()
            self.block1[0].bias.zero_()
            self.block1[0].weight[0, 0, :, 1] = 1.0 / 3.0
            self.block1[0].weight[1, 1, 1, :] = 1.0 / 3.0
            self.block1[0].weight[2, 2] = torch.eye(3) / 3.0
            self.block1[0].weight[3, :, 1, 1] = torch.tensor([0.25, 0.25, 0.25])

            self.block2[0].weight.zero_()
            self.block2[0].bias.zero_()
            self.block2[0].weight[0, 0, 1, 1] = 1.0
            self.block2[0].weight[1, 1, 1, 1] = 1.0
            self.block2[0].weight[2, 2, 1, 1] = 1.0
            self.block2[0].weight[3, 3, 1, 1] = 1.0
            self.block2[0].weight[4, 0, 1, 1] = 0.5
            self.block2[0].weight[4, 1, 1, 1] = 0.5
            self.block2[0].weight[5, 2, 1, 1] = 0.5
            self.block2[0].weight[5, 3, 1, 1] = 0.5

            self.head.weight.copy_(
                torch.tensor(
                    [
                        [1.0, 0.0, 0.0, 0.0, 0.2, 0.0],
                        [0.0, 1.0, 0.0, 0.0, 0.2, 0.0],
                        [0.0, 0.0, 1.0, 0.0, 0.0, 0.2],
                    ],
                    dtype=torch.float32,
                )
            )
            self.head.bias.zero_()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return class logits for an image batch."""

        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


def make_stimulus_images() -> list[Image.Image]:
    """Create eight deterministic PIL stimuli for MDS thumbnails."""

    images = []
    for index in range(8):
        image = Image.new("RGB", (32, 32), (12, 12, 12))
        draw = ImageDraw.Draw(image)
        color = [0, 0, 0]
        color[index % 3] = 210
        if index % 3 == 0:
            x_coord = 4 + (index // 3) * 8
            draw.rectangle((x_coord, 2, x_coord + 5, 29), fill=tuple(color))
        elif index % 3 == 1:
            y_coord = 4 + (index // 3) * 8
            draw.rectangle((2, y_coord, 29, y_coord + 5), fill=tuple(color))
        else:
            offset = (index // 3) * 3
            for delta in range(-2, 3):
                draw.line((2, 2 + offset + delta, 29, 29 + offset + delta), fill=tuple(color))
        draw.text((2, 2), str(index), fill=(255, 255, 255))
        images.append(image)
    return images


def image_batch_to_tensor(images: list[Image.Image]) -> torch.Tensor:
    """Convert a PIL image list into a float tensor batch."""

    tensors = []
    for image in images:
        array = np.asarray(image, dtype=np.float32) / 255.0
        tensors.append(torch.from_numpy(array).permute(2, 0, 1))
    return torch.stack(tensors)


model = TinyImageClassifier().eval()
images = make_stimulus_images()
mds_layers = tl.in_module("block1") | tl.in_module("block2")
print(f"stimuli={len(images)}; save selector={mds_layers}")

In [ ]:
trace = tl.trace(
    model,
    images,
    transform=image_batch_to_tensor,
    save=mds_layers,
    save_raw_input=True,
    random_seed=2024,
    output_style="classification",
)
trace.input_preprocessor = ResolvedPreprocessing(
    source="demo.image_batch_to_tensor",
    identifier="pil-rgb-32-float-v1",
    verified=True,
    config={"shape": [8, 3, 32, 32], "range": [0.0, 1.0]},
    description="PIL RGB image list -> rank-4 float tensor batch",
)

print("model_profile:")
print(trace.model_profile)
print("\noutput table, top-2 per stimulus:")
print(trace.output_table(top_n=2).to_string(index=False))
print("\nsummary(level='output'):")
print(trace.summary(level="output"))

In [ ]:
coords_by_layer = tl.repgeom.mds_evolution(trace, save=mds_layers, min_n=8)

print("MDS annotation blobs:")
for key, coords in coords_by_layer.items():
    print(f"  {key}: shape={coords.shape}, first=({coords[0, 0]:.4f}, {coords[0, 1]:.4f})")

In [ ]:
mds_svg = OUT / "semantic_io_keystone_mds"
trace.draw(
    vis_outpath=str(mds_svg),
    vis_save_only=True,
    vis_fileformat="svg",
    node_spec_fn=tl.repgeom.mds_scatter_node_spec(max_thumbnails=8),
)

input_svg = OUT / "semantic_io_keystone_input"
trace.draw(
    vis_outpath=str(input_svg),
    vis_save_only=True,
    vis_fileformat="svg",
    show_input_transform_summary=True,
)

print(f"MDS thumbnail graph: {mds_svg}.svg")
print(f"raw-input graph:      {input_svg}.svg")
print(f"annotation keys:      {sorted(trace._annotation_blobs or {})}")